# Task 3 — Generative AI Recommendation Articles with Qwen

This notebook uses `clustered_amazon_reviews.csv` from Task 2.

### Pipeline
**Task 2 categories → product statistics → weighted ranking → negative reviews → TF-IDF complaint themes → grounded evidence → Qwen2.5-1.5B-Instruct → recommendation articles → evaluation**

Make sure Task 2 contains meaningful `meta_category` names before running this notebook.

## 1. Imports and data

In [1]:
# Run once if required:
# !pip install -q -U transformers accelerate torch scikit-learn

import pandas as pd
import numpy as np
import re
import torch
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForCausalLM

df = pd.read_csv("clustered_reviews.csv", low_memory=False)

print("Shape:", df.shape)
display(df.head())

Shape: (60841, 11)


,id,product_name,name,brand,categories,source,reviews.rating,reviews.title,reviews.text,kmeans_cluster,kmeans_category
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,Kindle,This product so far has not disappointed. My c...,0,Fire Tablets
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,very fast,great for beginner or experienced person. Boug...,0,Fire Tablets
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,0,Fire Tablets
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,4.0,Good!!!,I've had my Fire HD 8 two weeks now and I love...,0,Fire Tablets
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,0,Fire Tablets


## 2. Validate and clean the Task 2 output

In [4]:
required = [
    "product_name", "kmeans_cluster", "kmeans_category",
    "reviews.text", "reviews.rating"
]

missing = [x for x in required if x not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

data = df.dropna(subset=required).copy()

if data["kmeans_category"].astype(str).str.contains(
    "RENAME_kmeans_cluster", na=False
).any():
    print("WARNING: Rename Task 2 kmeans_clusters before final generation.")

# Product names are cleaned in the next step.
# Keep the original `name` column here for traceability.
if "name" not in data.columns:
    raise ValueError("Missing required product column: name")

def clean_review(x):
    x = str(x).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", x).strip()

data["review_clean"] = data["reviews.text"].apply(clean_review)

print("Rows:", len(data))
print("categories:", data["kmeans_category"].nunique())
print("Products:", data["product_name"].nunique())

Rows: 55611
categories: 6
Products: 69


## 3. Clean Product Names

The raw Amazon `name` field can contain inconsistent spacing, HTML fragments, duplicated punctuation, and other formatting noise.

We keep the original product name and create a separate `product_name_clean` column for analysis.

This cleaning step does **not** affect Task 2 clustering; it is used only for Task 3 product statistics and recommendation output.


In [5]:
from html import unescape

def clean_product_name(name):
    if pd.isna(name):
        return np.nan

    name = unescape(str(name))

    # Remove HTML tags
    name = re.sub(r"<[^>]+>", " ", name)

    # Replace line breaks/tabs with spaces
    name = re.sub(r"[\n\r\t]+", " ", name)

    # Remove repeated punctuation
    name = re.sub(r"([,;:/|\\-])\1+", r"\1", name)

    # Remove spaces before punctuation
    name = re.sub(r"\s+([,;:])", r"\1", name)

    # Normalize whitespace
    name = re.sub(r"\s+", " ", name)

    # Remove leading/trailing punctuation and spaces
    name = name.strip(" ,;:-_|")

    return name.strip()


# Always create a clean product-name column from the original name
if "name" not in data.columns:
    raise ValueError(
        "The dataset does not contain the original 'name' column."
    )

data["product_name_clean"] = (
    data["name"]
    .apply(clean_product_name)
)

# Convert empty strings to NaN
data["product_name_clean"] = (
    data["product_name_clean"]
    .replace("", np.nan)
)

# Remove rows without a usable product name
before = len(data)

data = data.dropna(
    subset=["product_name_clean"]
).copy()

after = len(data)

# Use the cleaned name throughout Task 3
PRODUCT_COLUMN = "product_name_clean"

print("Rows removed because product name was empty:", before - after)
print("Unique original product names:", data["name"].nunique())
print("Unique cleaned product names:", data["product_name_clean"].nunique())

display(
    data[
        ["name", "product_name_clean"]
    ]
    .drop_duplicates()
    .head(20)
)


Rows removed because product name was empty: 1563
Unique original product names: 119
Unique cleaned product names: 114


,name,product_name_clean
0,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
2814,Kindle Oasis E-reader with Leather Charging Co...,Kindle Oasis E-reader with Leather Charging Co...
2881,"Amazon Kindle Lighted Leather Cover,,,\r\nAmaz...","Amazon Kindle Lighted Leather Cover, Amazon Ki..."
2883,"Amazon Kindle Lighted Leather Cover,,,\r\nKind...","Amazon Kindle Lighted Leather Cover, Kindle Ke..."
2884,"Kindle Keyboard,,,\r\nKindle Keyboard,,,","Kindle Keyboard, Kindle Keyboard"
2905,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,..."
3051,"Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ...","Fire HD 8 Tablet with Alexa, 8 HD Display, 32 ..."
3065,Amazon 5W USB Official OEM Charger and Power A...,Amazon 5W USB Official OEM Charger and Power A...
3266,"All-New Kindle E-reader - Black, 6 Glare-Free ...","All-New Kindle E-reader - Black, 6 Glare-Free ..."
3478,"Amazon Kindle Fire Hd (3rd Generation) 8gb,,,\...","Amazon Kindle Fire Hd (3rd Generation) 8gb, Am..."


## 4. Rating sentiment

We use the same rating interpretation as Task 1:

- 1–2 → negative
- 3 → neutral
- 4–5 → positive

In [6]:
def rating_sentiment(rating):
    if rating <= 2:
        return "negative"
    elif rating == 3:
        return "neutral"
    return "positive"

data["sentiment"] = data["reviews.rating"].apply(rating_sentiment)

display(data["sentiment"].value_counts().to_frame())

,count
sentiment,
positive,49500
neutral,2381
negative,2167


## 5. Category statistics

In [8]:
category_stats = (
    data.groupby("kmeans_category")
    .agg(
        review_count=("reviews.text", "count"),
        product_count=(PRODUCT_COLUMN, "nunique"),
        average_rating=("reviews.rating", "mean")
    )
    .sort_values("review_count", ascending=False)
)

display(category_stats.style.format({"average_rating": "{:.2f}"}))

,review_count,product_count,average_rating
kmeans_category,,,
Fire Tablets,29370,37,4.52
"Household, Office & Pets",12139,14,4.45
Smart Speakers & Echo,7423,29,4.67
E-readers & Kindle,5034,32,4.71
Chargers & Cables,56,8,4.23
Streaming & TV,26,6,4.27


## 6. Product statistics

In [9]:
product_stats = (
    data.groupby(["kmeans_category", PRODUCT_COLUMN])
    .agg(
        review_count=("reviews.rating", "count"),
        average_rating=("reviews.rating", "mean"),
        positive_reviews=("sentiment", lambda x: (x == "positive").sum()),
        neutral_reviews=("sentiment", lambda x: (x == "neutral").sum()),
        negative_reviews=("sentiment", lambda x: (x == "negative").sum())
    )
    .reset_index()
)

for s in ["positive", "neutral", "negative"]:
    product_stats[s + "_pct"] = (
        product_stats[s + "_reviews"]
        / product_stats["review_count"] * 100
    )

display(
    product_stats.sort_values("review_count", ascending=False).head(20)
)

,kmeans_category,product_name_clean,review_count,average_rating,positive_reviews,neutral_reviews,negative_reviews,positive_pct,neutral_pct,negative_pct
75,Fire Tablets,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",10751,4.451772,9745,683,323,90.642731,6.352897,3.004372
80,"Household, Office & Pets",AmazonBasics AAA Performance Alkaline Batterie...,8336,4.447817,7174,359,803,86.060461,4.306622,9.632917
79,"Household, Office & Pets",AmazonBasics AA Performance Alkaline Batteries...,3724,4.453276,3206,175,343,86.090226,4.699248,9.210526
23,E-readers & Kindle,Amazon Kindle Paperwhite - eBook reader - 4 GB...,2928,4.771858,2849,49,30,97.301913,1.673497,1.024590
113,Smart Speakers & Echo,"Echo (White), Echo (White)",2898,4.678054,2763,100,35,95.341615,3.450656,1.207729
47,Fire Tablets,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2814,4.586709,2667,93,54,94.776119,3.304904,1.918977
104,Smart Speakers & Echo,"Amazon Fire Tv, Amazon Fire Tv",2527,4.650178,2397,90,40,94.855560,3.561535,1.582905
64,Fire Tablets,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",2442,4.596233,2324,74,44,95.167895,3.030303,1.801802
45,Fire Tablets,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2275,4.587253,2153,77,45,94.637363,3.384615,1.978022
68,Fire Tablets,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",1680,4.513690,1527,91,62,90.892857,5.416667,3.690476


## 7. Weighted product ranking

Instead of ranking only by average rating, use a Bayesian-style weighted rating:

`weighted = (v/(v+m))*R + (m/(v+m))*C`

This reduces the chance that a product with only a few high ratings becomes the top recommendation.

In [14]:
C = data["reviews.rating"].mean()
m_reviews = max(3, int(product_stats["review_count"].quantile(0.25)))

product_stats["weighted_rating"] = (
    product_stats["review_count"]
    / (product_stats["review_count"] + m_reviews)
    * product_stats["average_rating"]
    +
    m_reviews
    / (product_stats["review_count"] + m_reviews)
    * C
)

def category_products(category):
    return product_stats[
        product_stats["kmeans_category"] == category
    ].copy()

def top_products(category, n=3):
    return (
        category_products(category)
        .sort_values(
            ["weighted_rating", "review_count"],
            ascending=[False, False]
        )
        .head(n)
    )

def weakest_product(category):
    p = category_products(category)
    if p.empty:
        return None

    return (
        p.sort_values(
            ["weighted_rating", "negative_pct"],
            ascending=[True, False]
        )
        .head(1)
    )

print("Global rating mean:", round(C, 3))
print("m:", m_reviews)

Global rating mean: 4.542
m: 3


## 8. Inspect ranked products

In [15]:
for category in data["kmeans_category"].drop_duplicates():

    print("\n" + "=" * 100)
    print(category)

    display(
        top_products(category)[
            [
                PRODUCT_COLUMN,
                "review_count",
                "average_rating",
                "weighted_rating",
                "positive_pct",
                "negative_pct"
            ]
        ]
    )


Fire Tablets


,product_name_clean,review_count,average_rating,weighted_rating,positive_pct,negative_pct
51,"All-New Fire HD 8 Tablet, 8"" HD Display, Wi-Fi...",3,5.00,4.771009,100.0,0.0
58,Amazon Kindle Touch Leather Case (4th Generati...,3,5.00,4.771009,100.0,0.0
53,Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Bl...,25,4.72,4.700931,96.0,0.0



E-readers & Kindle


,product_name_clean,review_count,average_rating,weighted_rating,positive_pct,negative_pct
24,Brand New Amazon Kindle Fire 16gb 7 Ips Displa...,59,4.830508,4.816549,100.000000,0.000000
29,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",52,4.826923,4.811383,98.076923,1.923077
27,Certified Refurbished Amazon Fire TV with Alex...,4,5.000000,4.803722,100.000000,0.000000



Streaming & TV


...

,product_name_clean,review_count,average_rating,weighted_rating,positive_pct,negative_pct
4,AmazonBasics USB 3.0 Cable - A-Male to B-Male ...,6,5.0,4.847339,100.0,0.0
2,AmazonBasics 16-Gauge Speaker Wire - 100 Feet,5,5.0,4.828257,100.0,0.0
6,Kindle PowerFast International Charging Kit (f...,5,4.8,4.703257,100.0,0.0



Smart Speakers & Echo


,product_name_clean,review_count,average_rating,weighted_rating,positive_pct,negative_pct
109,Certified Refurbished Amazon Echo,7,5.000000,4.862605,100.000,0.00000
112,"Echo (Black), Echo (Black)",3,5.000000,4.771009,100.000,0.00000
101,"Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",128,4.773438,4.768138,96.875,0.78125



Household, Office & Pets


,product_name_clean,review_count,average_rating,weighted_rating,positive_pct,negative_pct
89,Expanding Accordion File Folder Plastic Portab...,9,5.0,4.885505,100.0,0.0
88,Cat Litter Box Covered Tray Kitten Extra Large...,2,5.0,4.725211,100.0,0.0
84,AmazonBasics Nespresso Pod Storage Drawer - 50...,1,5.0,4.656514,100.0,0.0


# Complaint Analysis

## 9. Negative reviews

In [17]:
negative_reviews = (
    data[data["sentiment"] == "negative"]
    .drop_duplicates(
        subset=[
            "kmeans_category",
            PRODUCT_COLUMN,
            "review_clean"
        ]
    )
    .copy()
)

print("Negative reviews:", len(negative_reviews))

Negative reviews: 2161


## 10. Extract complaint themes with TF-IDF

The strongest words and bigrams from negative reviews become **candidate complaint themes**.

In [19]:
def extract_complaint_terms(category, n_terms=10):

    reviews = (
        negative_reviews[
            negative_reviews["kmeans_category"] == category
        ]["review_clean"]
        .drop_duplicates()
    )

    if len(reviews) < 2:
        return []

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95,
        sublinear_tf=True
    )

    X = vectorizer.fit_transform(reviews)

    terms = np.array(vectorizer.get_feature_names_out())
    scores = np.asarray(X.mean(axis=0)).ravel()

    top_indices = scores.argsort()[-n_terms:][::-1]

    return [
        (terms[i], float(scores[i]))
        for i in top_indices
    ]

for category in data["kmeans_category"].drop_duplicates():
    print("\n", category)
    print(extract_complaint_terms(category))


 Fire Tablets
[('tablet', 0.026468105571596488), ('amazon', 0.019908795714560928), ('bought', 0.015921032665102423), ('good', 0.015388684442699219), ('slow', 0.01537701113086478), ('apps', 0.015040591559904346), ('use', 0.014368936437912339), ('buy', 0.013492898275075686), ('like', 0.013285891585606164), ('screen', 0.01295801786857888)]

 E-readers & Kindle
[('kindle', 0.0336436967170532), ('buy', 0.02350508368482941), ('charge', 0.020102835458568055), ('read', 0.02004384710592899), ('books', 0.018992039690421535), ('charger', 0.017675273727569683), ('expensive', 0.016992545248853574), ('amazon', 0.01677311539450927), ('just', 0.016463990633299463), ('work', 0.015667256520710795)]

 Streaming & TV
[('amazon', 0.07461412820791982), ('refurbished', 0.07370504563840237), ('bought', 0.07211451893023704), ('box', 0.06784851644643981), ('tv', 0.06475627884193569), ('certified', 0.06346742732050384), ('boot', 0.05617197961301268), ('new', 0.04930719629598497), ('amazon logo', 0.0469986780850

## 11. Representative negative-review excerpts

In [21]:
def negative_examples(category, n=6):

    subset = negative_reviews[
        negative_reviews["kmeans_category"] == category
    ].copy()

    subset = subset[
        subset["review_clean"].str.len() >= 25
    ]

    subset["length"] = subset["review_clean"].str.len()

    return (
        subset.sort_values("length", ascending=False)
        .head(n)[
            [
                PRODUCT_COLUMN,
                "reviews.rating",
                "review_clean"
            ]
        ]
    )

for category in data["kmeans_category"].drop_duplicates():

    print("\n", category)

    display(
        negative_examples(category)
    )


 Fire Tablets


,product_name_clean,reviews.rating,review_clean
6383,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",1.0,"It's very appealing for the price, but know in..."
53237,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",1.0,I went with this tablet based on many good cus...
727,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",1.0,I went with this tablet based on many good cus...
60471,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",1.0,I went with this tablet based on many good cus...
13978,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",2.0,"We bought WowWee MiP robots for our grandsons,..."
14329,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",2.0,"I didn't expect a lot at this price, but I als..."



 E-readers & Kindle


,product_name_clean,reviews.rating,review_clean
18209,Amazon Kindle Paperwhite - eBook reader - 4 GB...,2.0,I wanted to like this a lot because my Ipad Mi...
15774,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",2.0,"I don't want to get too negative on this, beca..."
57178,All-New Kindle Oasis E-reader - 7 High-Resolut...,2.0,"I've always been a fan of the Kindle, ever sin..."
20837,"Kindle Voyage E-reader, 6 High-Resolution Disp...",1.0,"Very disappointed that my Kindle Voyage, purch..."
19734,Amazon Kindle Paperwhite - eBook reader - 4 GB...,2.0,I am on my 3rd Kindle. This software downloads...
20251,Amazon Kindle Paperwhite - eBook reader - 4 GB...,1.0,Used it for 2 days and the screen froze 5 time...



 Streaming & TV


...

,product_name_clean,reviews.rating,review_clean
49724,Amazon Kindle Charger Power Adapter Wall Charg...,2.0,"When I got my Kindle Fire, I got a protective ..."
49763,Oem Amazon Kindle Power Usb Adapter Wall Trave...,1.0,"No reason. This product is described as 5W, 1 ..."
49762,Oem Amazon Kindle Power Usb Adapter Wall Trave...,1.0,One star because this should've been included ...
58403,Amazon Kindle Replacement Power Adapter (Fits ...,2.0,I purchased my Kindle2 in March of 09 - and it...
58404,Amazon Kindle Replacement Power Adapter (Fits ...,1.0,Can someone explain to me why I can find hundr...
58401,Amazon Kindle Replacement Power Adapter (Fits ...,1.0,I was initially happy that my Kindle replaceme...



 Smart Speakers & Echo


,product_name_clean,reviews.rating,review_clean
27374,"Amazon Fire Tv, Amazon Fire Tv",1.0,Bestbuy came through before the holiday better...
26896,"Amazon Fire Tv, Amazon Fire Tv",2.0,I received this as a Christmas gift. I was exc...
23477,"Echo (White), Echo (White)",2.0,I bought this to fill my life with music. The ...
25313,Amazon - Amazon Tap Portable Bluetooth and Wi-...,1.0,I was excited about the Amazon echo. However a...
49742,Amazon Tap Smart Assistant Alexaenabled (black...,2.0,"Like DOS, you need to memorize your command an..."
27162,"Amazon Fire Tv, Amazon Fire Tv",1.0,I purchased this item on the recommendation of...



 Household, Office & Pets


,product_name_clean,reviews.rating,review_clean
48763,AmazonBasics AA Performance Alkaline Batteries...,2.0,I am extremely disappointed with these batteri...
42379,AmazonBasics AAA Performance Alkaline Batterie...,2.0,"So, I purchased a pack of these a month ago (1..."
48704,AmazonBasics AA Performance Alkaline Batteries...,2.0,"A day or two after these batteries arrived, I ..."
43773,AmazonBasics AAA Performance Alkaline Batterie...,2.0,"A day or two after these batteries arrived, I ..."
38302,AmazonBasics AAA Performance Alkaline Batterie...,1.0,These are the WORST batteries I have ever enco...
45562,AmazonBasics AAA Performance Alkaline Batterie...,1.0,I came here to look at the one and two star re...


# Build Evidence for the LLM

## 12. Structured evidence package

In [22]:
def build_evidence(category):

    subset = data[
        data["kmeans_category"] == category
    ]

    return {
        "category": category,
        "review_count": len(subset),
        "product_count": subset[PRODUCT_COLUMN].nunique(),
        "average_rating": subset["reviews.rating"].mean(),
        "top_products": top_products(category, 3),
        "weakest": weakest_product(category),
        "complaint_terms": extract_complaint_terms(category, 10),
        "negative_examples": negative_examples(category, 6)
    }

evidence = {
    category: build_evidence(category)
    for category in data["kmeans_category"].drop_duplicates()
}

example_category = next(iter(evidence))

print("Evidence packages:", len(evidence))
print("Example category:", example_category)

Evidence packages: 6
Example category: Fire Tablets


## 13. Grounded prompt

In [23]:
def build_prompt(e):

    lines = [
        "Write a detailed consumer recommendation article from the Amazon review evidence below.",
        "",
        "STRICT RULES:",
        "- Use ONLY the supplied evidence.",
        "- Never invent specifications, features, prices or complaints.",
        "- Keep all numerical claims consistent with the supplied data.",
        "- Treat TF-IDF complaint terms as candidate themes and verify them against review excerpts.",
        "- If evidence is insufficient, explicitly say so.",
        "",
        f"CATEGORY: {e['category']}",
        (
            f"DATA: {e['product_count']} products; "
            f"{e['review_count']} reviews; "
            f"category average rating={e['average_rating']:.2f}/5."
        ),
        "",
        "TOP PRODUCTS:"
    ]

    for rank, (_, row) in enumerate(
        e["top_products"].iterrows(), 1
    ):
        lines.append(
            f"{rank}. {row[PRODUCT_COLUMN]} | "
            f"reviews={int(row['review_count'])} | "
            f"rating={row['average_rating']:.2f}/5 | "
            f"weighted rating={row['weighted_rating']:.2f}/5 | "
            f"positive={row['positive_pct']:.1f}% | "
            f"negative={row['negative_pct']:.1f}%"
        )

    weakest = e["weakest"]

    if weakest is not None and not weakest.empty:
        row = weakest.iloc[0]

        lines += [
            "",
            "WEAKEST PRODUCT BY WEIGHTED EVIDENCE:",
            (
                f"{row[PRODUCT_COLUMN]} | "
                f"reviews={int(row['review_count'])} | "
                f"rating={row['average_rating']:.2f}/5 | "
                f"weighted rating={row['weighted_rating']:.2f}/5 | "
                f"negative={row['negative_pct']:.1f}%"
            )
        ]

    lines += [
        "",
        "CANDIDATE COMPLAINT THEMES:"
    ]

    if e["complaint_terms"]:
        for term, score in e["complaint_terms"]:
            lines.append(f"- {term}")
    else:
        lines.append("- insufficient negative-review evidence")

    lines += [
        "",
        "NEGATIVE REVIEW EVIDENCE:"
    ]

    for _, row in e["negative_examples"].iterrows():
        lines.append(
            f"- {row[PRODUCT_COLUMN]} "
            f"({row['reviews.rating']}/5): "
            f"{row['review_clean'][:450]}"
        )

    lines += [
        "",
        "WRITE EXACTLY THESE SECTIONS:",
        "## Category Overview",
        "## Top 3 Products",
        "## Key Differences",
        "## Common Customer Complaints",
        "## Weakest Product",
        "## Final Recommendation",
        "",
        "Explain conclusions with the supplied evidence.",
        "Aim for approximately 400-600 words."
    ]

    return "\n".join(lines)

print(build_prompt(evidence[example_category]))

Write a detailed consumer recommendation article from the Amazon review evidence below.

STRICT RULES:
- Use ONLY the supplied evidence.
- Never invent specifications, features, prices or complaints.
- Keep all numerical claims consistent with the supplied data.
- Treat TF-IDF complaint terms as candidate themes and verify them against review excerpts.
- If evidence is insufficient, explicitly say so.

CATEGORY: Fire Tablets
DATA: 37 products; 29370 reviews; category average rating=4.52/5.

TOP PRODUCTS:
1. All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta | reviews=3 | rating=5.00/5 | weighted rating=4.77/5 | positive=100.0% | negative=0.0%
2. Amazon Kindle Touch Leather Case (4th Generation - 2011 Release), Olive Green, Amazon Kindle Touch Leather Case (4th Generation - 2011 Release), Olive Green | reviews=3 | rating=5.00/5 | weighted rating=4.77/5 | positive=100.0% | negative=0.0%
3. Amazon - Kindle Voyage - 4GB - Wi-Fi + 3G - Black, Amazon - K

# Generative AI

## 14. Load Qwen2.5-1.5B-Instruct

This replaces FLAN-T5 and does not use `text2text-generation`.

In [24]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

model.eval()

print("Loaded:", MODEL_NAME)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


## 15. Article generation function

In [25]:
def generate_article(category, max_new_tokens=700):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful product-review analyst. "
                "Use only supplied evidence. "
                "Never invent unsupported product facts."
            )
        },
        {
            "role": "user",
            "content": build_prompt(
                evidence[category]
            )
        }
    ]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

## 16. Test one category first

In [ ]:
print("Generating:", example_category)

example_article = generate_article(
    example_category
)

print("\n" + "=" * 100)
print(example_article)

## 17. Generate all category articles

In [26]:
generated = []

for category in evidence:

    print("Generating:", category)

    try:
        article = generate_article(category)
        status = "success"

    except Exception as error:
        article = ""
        status = str(error)

    generated.append({
        "kmeans_category": category,
        "generated_article": article,
        "status": status
    })

articles = pd.DataFrame(generated)

display(articles)

Generating: Fire Tablets
Generating: E-readers & Kindle
Generating: Streaming & TV
Generating: Chargers & Cables
Generating: Smart Speakers & Echo
Generating: Household, Office & Pets


,kmeans_category,generated_article,status
0,Fire Tablets,### Category Overview\n\nThe Fire Tablets cate...,success
1,E-readers & Kindle,### Category Overview\nThe Amazon Kindle serie...,success
2,Streaming & TV,### Category Overview\nThe streaming and telev...,success
3,Chargers & Cables,### Category Overview\n\nThe Amazon review dat...,success
4,Smart Speakers & Echo,### Category Overview\n\nThe Smart Speakers & ...,success
5,"Household, Office & Pets","### Category Overview\nThe household, office, ...",success


## 18. Inspect all generated articles

In [27]:
for _, row in articles.iterrows():

    print("\n" + "=" * 110)
    print(row["kmeans_category"])
    print("=" * 110)

    print(row["generated_article"])


Fire Tablets
### Category Overview

The Fire Tablets category offers a range of options from different manufacturers, catering to various user preferences and requirements. The average rating across 37 products reviewed is 4.52/5, indicating generally high satisfaction among users. This suggests that the majority of customers find value in these devices, although there are notable exceptions.

### Top 3 Products

#### 1. All-New Fire HD 8 Tablet, 8" HD Display, Wi-Fi, 16 GB - Includes Special Offers, Magenta
**Reviews:** 3  
**Rating:** 5.00/5  
**Weighted Rating:** 4.77/5  
**Positive:** 100.0%  
**Negative:** 0.0%

This product stands out as the highest-rated option, receiving perfect scores across all metrics. Customers appreciate its 8-inch display, Wi-Fi connectivity, and ample storage capacity. The inclusion of special offers adds convenience for potential buyers.

#### 2. Amazon Kindle Touch Leather Case (4th Generation - 2011 Release), Olive Green
**Reviews:** 3  
**Rating:** 

## 19. Save the results

In [29]:
articles.to_csv(
    "category_recommendation_articles.csv",
    index=False
)

parts = [
    "# Amazon Product Review Recommendations\n"
]

for _, row in articles.iterrows():

    parts.append(
        f"## {row['kmeans_category']}\n"
    )

    parts.append(
        row["generated_article"] + "\n"
    )

Path(
    "product_recommendation_articles.md"
).write_text(
    "\n".join(parts),
    encoding="utf-8"
)

print("Saved category_recommendation_articles.csv")
print("Saved product_recommendation_articles.md")

Saved category_recommendation_articles.csv
Saved product_recommendation_articles.md


# Final Task 3 Methodology

```text
Task 2 meta-categories
        ↓
Category / product statistics
        ↓
Bayesian weighted product ranking
        ↓
Negative reviews
        ↓
TF-IDF complaint-theme extraction
        ↓
Representative complaint excerpts
        ↓
Structured evidence
        ↓
Grounded prompt
        ↓
Qwen2.5-1.5B-Instruct
        ↓
400–600 word recommendation articles
```

This design lets traditional NLP/statistical methods establish the evidence first and uses generative AI primarily for synthesis and readable recommendation writing.